# REINFORCE et REINFORCE with Baseline sur CartPole : les premiers policy gradients

**Algorithmes** : REINFORCE et REINFORCE with Baseline, construits en parallele puis compare  
**Environnement** : CartPole-v1 (Gymnasium)  
**Papiers** : [Williams, 1992](https://link.springer.com/article/10.1007/BF00992696) (REINFORCE) ; [Sutton et al., 2000](https://papers.nips.cc/paper/1713-policy-gradient-methods-for-reinforcement-learning-with-function-approximation) (Policy Gradient theorem)

| Propriete | REINFORCE | REINFORCE + Baseline |
|-----------|-----------|----------------------|
| Model-free | oui | oui |
| On-policy | oui | oui |
| Policy-based | oui | oui |
| Monte Carlo | oui | oui |
| Replay buffer | non | non |
| Reseau de valeur | non | oui |
| Variance du gradient | elevee | reduite |

Dans les notebooks precedents, on a construit des methodes **value-based** : Q-learning tabulaire, puis DQN et Double DQN. Ces methodes apprennent $Q(s, a)$, la valeur de chaque paire (etat, action), et en deduisent la politique en prenant le $\arg\max$.

Ce notebook marque un **changement de paradigme** : on passe aux methodes **policy gradient**. Plutot d'apprendre une fonction de valeur puis d'en deduire la politique, on parametre la politique directement et on l'optimise par montee de gradient sur la recompense esperee.

REINFORCE et REINFORCE with Baseline partagent **tout** : le reseau de politique, la selection d'action stochastique, les returns Monte Carlo, la formule du gradient. La seule difference est que la version avec baseline soustrait une estimation de $V(s)$ pour reduire la variance du gradient.

## Pourquoi optimiser la politique directement

### Le probleme avec les methodes value-based

L'approche value-based fonctionne en deux etapes : apprendre $Q(s, a)$, puis choisir $\pi(s) = \arg\max_a Q(s, a)$. Ca marche tres bien pour les actions discretes en petit nombre (pousser a gauche ou a droite sur CartPole).

Mais le $\arg\max$ est **intractable pour les actions continues**. Imagine un bras robotique avec 7 degrés de liberte et des couples en valeurs reelles : enumerer toutes les actions possibles est impossible. Et meme en discretisant grossierement, on explose combinatoirement.

De plus, l'approche value-based force la politique a etre **deterministe** : toujours l'action avec la plus haute Q-valeur. Or certains jeux necessitent une politique **stochastique** (ex: pierre-feuille-ciseaux : si l'adversaire peut predire ta strategie, il gagne toujours).

### La solution : parametrer la politique directement

On parametre la politique directement : $\pi(a | s; \theta)$. Le reseau prend un etat et produit une **distribution de probabilite sur les actions**. On optimise $\theta$ par montee de gradient sur la recompense esperee :

$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

**Avantages des policy gradients :**
- Actions continues : la politique peut parametrer une gaussienne $\mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$
- Politiques stochastiques naturelles, utiles dans les jeux et environnements partiellement observables
- Distributions multimodales : plusieurs bonnes actions possibles simultanement
- Convergence vers un optimum local garantie sous conditions

**Prix a payer :**
- **Variance elevee** du gradient : c'est le probleme central de ce notebook
- **On-policy** : pas de replay buffer, chaque experience ne sert qu'une fois
- Convergence plus lente que DQN sur des problemes simples
- Sensible au choix du taux d'apprentissage

**Analogie.** Les methodes value-based, c'est comme apprendre la note de chaque plat dans tous les restaurants d'une ville pour toujours choisir le meilleur. Policy gradient, c'est apprendre directement a commander : tu essaies des plats, tu ajustes tes preferences selon ta satisfaction, sans jamais construire de guide Michelin interne.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium.wrappers import RecordVideo
from torch.distributions import Categorical

try:
    from IPython.display import Video, display
except ImportError:
    Video = None
    display = None

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100


## CartPole-v1 (rappel rapide)

On reutilise le meme environnement. L'observation est un vecteur continu de 4 dimensions, l'espace d'actions est discret avec 2 actions.

| Dimension | Variable | Plage |
|-----------|----------|-------|
| 0 | Position du chariot | $[-4.8,\ 4.8]$ |
| 1 | Vitesse du chariot | $(-\infty,\ +\infty)$ |
| 2 | Angle du pendule | $[-0.418,\ 0.418]$ rad |
| 3 | Vitesse angulaire | $(-\infty,\ +\infty)$ |

| Action | Signification |
|--------|---------------|
| 0 | Pousser le chariot a gauche |
| 1 | Pousser le chariot a droite |

Recompense : $+1$ a chaque pas. Score maximal : 500. Episode termine si le pendule depasse $\pm 12$ degres ou si le chariot sort de la zone.

In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=42)

print(f"Espace d'observation : {env.observation_space}")
print(f"  Dimension : {env.observation_space.shape[0]}")
print(f"\nEspace d'actions : {env.action_space}")
print(f"  Nombre d'actions : {env.action_space.n}")
print(f"\nObservation initiale : {obs}")

# Un episode aleatoire
obs, _ = env.reset(seed=42)
total_reward, steps, done = 0, 0, False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

print(f"\nEpisode aleatoire : {steps} pas, recompense = {total_reward}")
env.close()

## Le reseau de politique

### Difference fondamentale avec QNetwork

Dans DQN, le reseau $Q(s, \cdot; \theta) : \mathbb{R}^4 \to \mathbb{R}^2$ produit une **valeur** par action. L'agent choisit l'action avec la valeur maximale.

Le reseau de politique $\pi(\cdot | s; \theta) : \mathbb{R}^4 \to \Delta^{|A|}$ produit une **distribution de probabilite** sur les actions. L'agent **echantillonne** depuis cette distribution.

### Architecture

$$\mathrm{obs}\in\mathbb{R}^{4}\xrightarrow{\ \mathrm{Linear}(4,64)\ }\mathbb{R}^{64}\xrightarrow{\ \mathrm{ReLU}\ }\mathbb{R}^{64}\xrightarrow{\ \mathrm{Linear}(64,64)\ }\mathbb{R}^{64}\xrightarrow{\ \mathrm{ReLU}\ }\mathbb{R}^{64}\xrightarrow{\ \mathrm{Linear}(64,2)\ }\mathrm{logits}\in\mathbb{R}^{2}$$

La couche de sortie produit des **logits** (scores bruts non normalises), pas directement des probabilites. La distribution `Categorical(logits=...)` applique un softmax en interne.

**Pourquoi des logits et pas des probabilites (softmax en sortie) ?**

Stabilite numerique. On a besoin de $\log \pi(a|s; \theta)$ pour calculer le gradient (voir plus bas). Si on applique softmax en sortie puis $\log$, on obtient des instabilites numeriques ($\log(\text{tres petite proba}) \to -\infty$). `Categorical` utilise `log_softmax` en interne, qui est numeriquement stable.

### Pas de $\arg\max$ : on **echantillonne**

C'est le changement conceptuel central. Au lieu de choisir deterministement l'action maximale, on tire une action depuis $\pi(\cdot | s; \theta)$. L'exploration est **inherente** a la politique, pas ajoutee artificiellement via $\varepsilon$-greedy.

In [ ]:
class PolicyNetwork(nn.Module):
    """Reseau de politique pi(a|s; theta) : observation -> logits des actions."""

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # logits bruts, pas de softmax


# --- Test sur une observation ---
torch.manual_seed(42)
obs_dim, n_actions = 4, 2
policy_net = PolicyNetwork(obs_dim, n_actions)

obs_tensor = torch.tensor([0.02, -0.01, 0.04, 0.02], dtype=torch.float32)
logits = policy_net(obs_tensor)
probs = torch.softmax(logits, dim=-1)

print(f"Observation : {obs_tensor.tolist()}")
print(f"Logits      : [gauche={logits[0].item():.4f}, droite={logits[1].item():.4f}]")
print(f"Probabilites: [gauche={probs[0].item():.4f}, droite={probs[1].item():.4f}]")
print(f"Somme       : {probs.sum().item():.6f}  (doit etre 1.0)")
print(f"\nParametres totaux : {sum(p.numel() for p in policy_net.parameters()):,}")

# Test batch
batch = torch.randn(32, obs_dim)
logits_batch = policy_net(batch)
print(f"\nBatch : {batch.shape} -> logits : {logits_batch.shape}")

## Selection d'action stochastique

### Fini l'$\varepsilon$-greedy

Avec DQN, la politique est deterministe ($\arg\max Q$) et on ajoute une exploration artificielle via $\varepsilon$-greedy. C'est un bricolage : la politique apprise est greedy, mais on la force a etre aleatoire pendant l'entrainement.

Avec REINFORCE, la politique **est** stochastique par construction. $\pi(a | s; \theta)$ est une vraie distribution de probabilite. Pas besoin de $\varepsilon$-greedy : l'exploration emerge naturellement des probabilites de la politique.

### La distribution `Categorical`

`Categorical(logits=logits)` cree une distribution discrete sur $|A|$ actions. Deux operations essentielles :

- `.sample()` : tire une action proportionnellement aux probabilites. Utilise pendant l'**entrainement** pour explorer.
- `.log_prob(action)` : retourne $\log \pi(a | s; \theta)$. **Crucial** : c'est ce terme qui porte le gradient vers les parametres $\theta$.

On sauvegarde `log_prob(action)` a chaque pas de l'episode, car on en aura besoin pour calculer la loss apres la fin de l'episode (Monte Carlo : on attend la fin pour connaitre les returns).

**En phase d'exploitation** (apres entrainement), on peut prendre $\arg\max$ pour une politique deterministe, ou continuer a echantillonner.

In [ ]:
def select_action(policy_net, state):
    """Echantillonne une action depuis pi(.|s; theta) et retourne son log_prob."""
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)  # (1, obs_dim)
    logits = policy_net(state_t)                                       # (1, n_actions)
    dist = Categorical(logits=logits)
    action = dist.sample()                    # scalaire entier
    log_prob = dist.log_prob(action)          # scalaire, grad_fn=LogSoftbackward
    return action.item(), log_prob


# --- Demo : distribution des actions pour une observation fixe ---
torch.manual_seed(42)
policy_net = PolicyNetwork(obs_dim, n_actions)
obs = np.array([0.02, -0.01, 0.04, 0.02])

# Distribution theorique
with torch.no_grad():
    logits = policy_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
    probs = torch.softmax(logits, dim=-1).squeeze()
print(f"Distribution theorique : gauche={probs[0].item():.4f}, droite={probs[1].item():.4f}")

# Distribution empirique sur 10 000 tirages
counts = {0: 0, 1: 0}
for _ in range(10_000):
    a, _ = select_action(policy_net, obs)
    counts[a] += 1

print(f"Distribution empirique : gauche={counts[0]/100:.2f}%, droite={counts[1]/100:.2f}%")
print(f"(sur 10 000 tirages, attendu ~{probs[0].item()*100:.1f}% gauche)")

# Le log_prob porte bien le gradient
action, log_prob = select_action(policy_net, obs)
print(f"\nAction tiree : {action}")
print(f"log_prob : {log_prob.item():.4f}  (== log(prob de l'action choisie))")
print(f"grad_fn  : {log_prob.grad_fn}  (non-None = gradient disponible)")

## Les returns Monte Carlo

### On-policy et Monte Carlo : on attend la fin

REINFORCE est un algorithme **Monte Carlo** : on joue un episode complet, on observe toutes les recompenses, puis on calcule les returns et on met a jour la politique. Pas de bootstrap (pas de cible TD). On utilise les vraies recompenses cumulees.

Le return depuis le pas $t$ est :

$$G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots + \gamma^{T-t} r_T$$

### Calcul efficace : recurrence backward

On calcule $G_t$ en partant de la fin de l'episode :

$$G_T = r_T, \quad G_{t} = r_t + \gamma G_{t+1}$$

C'est $O(T)$ au lieu de $O(T^2)$ si on calculait chaque $G_t$ independamment.

### Normalisation des returns

On normalise les returns : $\hat{G}_t = \frac{G_t - \mu}{\sigma + \epsilon}$ avec $\mu = \text{mean}(\{G_t\})$ et $\sigma = \text{std}(\{G_t\})$.

Pourquoi ? Les returns bruts peuvent etre dans une plage tres large (1 a 500 sur CartPole). Normaliser :
- Stabilise les gradients (meme echelle d'un episode a l'autre)
- Certains $G_t$ deviennent negatifs (en dessous de la moyenne) : l'agent est **penalise** pour les actions prises dans des pas au-dessous de la moyenne
- Certains $G_t$ sont positifs : l'agent est **encourage** pour ces actions

C'est une forme implicite de baseline (centre les returns autour de zero).

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """Calcule les returns Monte Carlo normalises depuis la fin."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns

def compute_raw_returns(rewards, gamma=0.99):
    """Calcule les returns Monte Carlo SANS normalisation (cibles pour V(s))."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return torch.tensor(returns, dtype=torch.float32)


rewards_demo = [1.0] * 10
gamma_demo = 0.99
G0_manuel = sum(gamma_demo**k * r for k, r in enumerate(rewards_demo))
print("Episode fictif (10 pas, r=1 partout, gamma=0.99) :")
print(f"  G_0 manuel : {G0_manuel:.6f}")

returns_demo = compute_returns(rewards_demo, gamma_demo)
raw_returns_demo = compute_raw_returns(rewards_demo, gamma_demo)
print(f"  Returns bruts : G_0={raw_returns_demo[0].item():.4f}, ..., G_9={raw_returns_demo[-1].item():.4f}")
print(f"  Returns normalises : {returns_demo.tolist()}")
print(f"  Mean={returns_demo.mean().item():.6f}, Std={returns_demo.std().item():.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
steps = list(range(10))
ax1.bar(steps, raw_returns_demo.numpy(), color="steelblue")
ax1.set_title("Returns bruts G_t")
ax1.set_xlabel("Pas t")
ax1.set_ylabel("G_t")
ax1.grid(True, alpha=0.3)
ax2.bar(steps, returns_demo.numpy(), color="coral")
ax2.axhline(0, color="black", linestyle="--", linewidth=1)
ax2.set_title("Returns normalises pour la policy")
ax2.set_xlabel("Pas t")
ax2.set_ylabel("G_t normalise")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

La normalisation
$$
\hat{G}_t = \frac{G_t-\mu}{\sigma+\varepsilon}
$$
centre les returns autour de zéro pour la **mise à jour de la politique**. En revanche, lorsque l'on ajoute une baseline apprise, le réseau de valeur doit viser les **returns bruts**
$$
G_t=\sum_{k=0}^{T-t}\gamma^k r_{t+k}
$$
car il doit apprendre la bonne échelle de retour plutôt qu'une quantité déjà recentrée.

## Le theoreme du gradient de politique

### Objectif

On veut maximiser la recompense totale esperee :

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[G_0\right] = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]$$

ou $\tau = (s_0, a_0, r_0, s_1, a_1, \ldots)$ est une trajectoire echantillonnee depuis $\pi_\theta$.

### Le probleme : gradient d'une esperance

$$\nabla_\theta J(\theta) = \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[G_0]$$

Le gradient entre dans l'esperance, mais l'esperance depend elle-meme de $\theta$ (via la distribution $\pi_\theta$). On ne peut pas simplement echantillonner et faire backprop directement.

### Le log-trick (REINFORCE)

On utilise l'identite $\nabla_\theta \log f = \frac{\nabla_\theta f}{f}$, donc $\nabla_\theta f = f \cdot \nabla_\theta \log f$ :

$$\begin{aligned}
\nabla_\theta J(\theta) &= \nabla_\theta \sum_\tau p(\tau; \theta) G(\tau) \\
&= \sum_\tau \underbrace{p(\tau; \theta) \nabla_\theta \log p(\tau; \theta)}_{\text{log-trick}} G(\tau) \\
&= \mathbb{E}_{\tau \sim \pi_\theta}\left[ G(\tau) \nabla_\theta \log p(\tau; \theta) \right]
\end{aligned}$$

Or $\log p(\tau; \theta) = \log p(s_0) + \sum_t \log \pi_\theta(a_t | s_t) + \log p(s_{t+1} | s_t, a_t)$. Les termes de dynamique $\log p(s_{t+1} | s_t, a_t)$ ne dependent pas de $\theta$ et disparaissent au gradient :

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} G_t \cdot \nabla_\theta \log \pi_\theta(a_t | s_t) \right]}$$

**Intuition du chef cuisinier.** Le gradient dit : si la recette a produit un bon repas ($G_t > 0$), renforce les ingredients qui ont ete utilises ($\nabla_\theta \log \pi$). Si le repas etait mauvais ($G_t < 0$ apres normalisation), penalise ces ingredients. L'amplitude du renforcement est proportionnelle a la qualite du repas.

### Loss PyTorch (maximisation -> minimisation)

PyTorch minimize par defaut. On inverse le signe :

$$\mathcal{L}(\theta) = -\frac{1}{T} \sum_{t=0}^{T} \underbrace{\log \pi_\theta(a_t | s_t)}_{\text{log\_prob}} \cdot \underbrace{G_t}_{\text{return normalise}}$$

Minimiser $\mathcal{L}$ revient a maximiser $J(\theta)$.

In [ ]:
# --- Demo du gradient sur un episode ---
torch.manual_seed(42)
policy_net = PolicyNetwork(obs_dim=4, n_actions=2)
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)

env_demo = gym.make("CartPole-v1")
obs, _ = env_demo.reset(seed=42)

log_probs, rewards = [], []
done = False

# Collecter un episode
while not done:
    action, log_prob = select_action(policy_net, obs)
    obs, reward, terminated, truncated, _ = env_demo.step(action)
    log_probs.append(log_prob)
    rewards.append(reward)
    done = terminated or truncated

env_demo.close()
print(f"Episode : {len(rewards)} pas, recompense = {sum(rewards):.0f}")

# Calculer les returns
returns = compute_returns(rewards, gamma=0.99)

# Probabilite de l'action 0 AVANT la mise a jour
obs_fixed = torch.tensor([0.02, -0.01, 0.04, 0.02], dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    probs_before = torch.softmax(policy_net(obs_fixed), dim=-1)

# Calcul de la loss REINFORCE
log_probs_tensor = torch.stack(log_probs).view(-1)
assert log_probs_tensor.shape == returns.shape
loss = -(log_probs_tensor * returns).mean()

# Gradient step
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Probabilite de l'action 0 APRES la mise a jour
with torch.no_grad():
    probs_after = torch.softmax(policy_net(obs_fixed), dim=-1)

print(f"\nLoss REINFORCE : {loss.item():.6f}")
print(f"\nChangement des probabilites pour obs=[0.02, -0.01, 0.04, 0.02] :")
print(f"  Avant : gauche={probs_before[0,0].item():.4f}, droite={probs_before[0,1].item():.4f}")
print(f"  Apres : gauche={probs_after[0,0].item():.4f}, droite={probs_after[0,1].item():.4f}")

## Le probleme de la variance

### D'ou vient la variance ?

REINFORCE est un estimateur Monte Carlo du gradient. Comme tout estimateur Monte Carlo, il est **non biaise** : en moyenne sur de nombreux episodes, il estime le vrai gradient. Mais il a une **variance elevee**.

Deux sources de variance :

1. **La stochasticite de l'environnement** : le meme etat peut mener a des transitions differentes.
2. **La stochasticite de la politique** : pour le meme etat, l'agent tire des actions differentes. Des trajectoires tres differentes donnent des returns tres differents, donc des gradients tres differents.

Concretement : un episode peut se terminer apres 5 pas (return = 5) ou 200 pas (return = 200). Ces deux experiences poussent le gradient dans la meme direction (les deux sont positifs, l'agent a survace), mais avec des amplitudes completement differentes. Le gradient "saute" partout.

### Consequence pratique

Avec une variance elevee, l'apprentissage est erratique. La courbe de recompense monte et descend violemment. Il faut beaucoup d'episodes pour voir une tendance claire. Le taux d'apprentissage doit etre petit pour eviter que les mises a jour bruitees ne fassent partir les poids dans de mauvaises directions.

### La solution : une baseline

On peut soustraire une **baseline** $b(s_t)$ au return sans introduire de biais :

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ \sum_{t=0}^{T} (G_t - b(s_t)) \cdot \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

Preuve : $\mathbb{E}[b(s_t) \nabla_\theta \log \pi_\theta(a_t | s_t)] = b(s_t) \mathbb{E}[\nabla_\theta \log \pi_\theta(a_t | s_t)] = 0$ car $\mathbb{E}[\nabla_\theta \log \pi] = 0$ (propriete fondamentale des distributions de probabilite). La baseline ne biaise pas le gradient, mais elle peut reduire sa variance.

## Le reseau de valeur $V(s)$ comme baseline

### Quelle baseline choisir ?

La baseline optimale (en termes de reduction de variance) est $b^*(s_t) = \mathbb{E}[G_t | s_t] = V^\pi(s_t)$ : la valeur esperee de l'etat sous la politique courante. Intuitivement : on ne veut pas renforcer une action parce que le return global est positif, mais parce que le return est **meilleur que d'habitude** depuis cet etat.

On approxime $V^\pi(s)$ avec un second reseau de neurones $V(s; \phi)$, appris en parallele.

### L'avantage

La quantite $(G_t - V(s_t; \phi))$ s'appelle l'**avantage** de l'action $a_t$ :

$$A_t = G_t - V(s_t; \phi)$$

$$\begin{cases}
A_t > 0 & \text{: cette action a fait mieux que prevu depuis } s_t \to \text{renforcer} \\
A_t < 0 & \text{: cette action a fait moins bien que prevu} \to \text{penaliser} \\
A_t \approx 0 & \text{: l'action est dans la moyenne, signal faible}
\end{cases}$$

**Analogie.** Un etudiant qui obtient 14/20 a un examen : est-ce bien ou mal ? Ca depend. Si la moyenne de la classe est 10/20, c'est excellent (avantage positif). Si la moyenne est 17/20, c'est en dessous des attentes (avantage negatif). La baseline, c'est la moyenne de la classe.

### Entrainement du reseau de valeur

On entraine $V(s; \phi)$ a predire $G_t$ par regression :

$$\mathcal{L}_V(\phi) = \frac{1}{T} \sum_{t=0}^{T} \left( G_t - V(s_t; \phi) \right)^2$$

Le gradient de politique devient :

$$\mathcal{L}_{\pi}(\theta) = -\frac{1}{T} \sum_{t=0}^{T} \log \pi_\theta(a_t | s_t) \cdot \underbrace{(G_t - V(s_t; \phi))}_{\text{avantage}}$$

Deux reseaux, deux optimiseurs, deux losses. L'avantage est detache du graphe de calcul de la loss de valeur ($G_t - V(s_t)$ : on n'entraine pas $\phi$ via la loss de politique).

## Les deux gradients cote a cote

Les deux algorithmes partagent la même trajectoire, mais pas la même pondération du gradient :

```mermaid
flowchart LR
classDef default fill:none,stroke:#64748b,stroke-width:1.5px
classDef primary fill:none,stroke:#2563eb,stroke-width:2px
classDef secondary fill:none,stroke:#d97706,stroke-width:2px
classDef result fill:none,stroke:#16a34a,stroke-width:2px
classDef warning fill:none,stroke:#dc2626,stroke-width:2px
T["Trajectoire complete"]:::primary --> R["Returns bruts"]:::primary
R --> P["Returns normalises pour la policy"]:::secondary
R --> V["Baseline V(s) sur l'echelle brute"]:::secondary
P --> G1["Gradient REINFORCE"]:::result
V --> A["Avantage G - V(s)"]:::result
A --> G2["Gradient avec baseline"]:::result
```

Autrement dit : la normalisation reste une aide pour la politique, tandis que le critique apprend toujours la grandeur brute à prédire.

In [ ]:
class ValueNetwork(nn.Module):
    """Reseau de valeur V(s; phi) : observation -> scalaire."""

    def __init__(self, obs_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x).squeeze(-1)


def reinforce_loss(log_probs, returns):
    log_probs_tensor = torch.stack(log_probs).view(-1)
    assert log_probs_tensor.shape == returns.shape
    return -(log_probs_tensor * returns).mean()


def reinforce_baseline_loss(log_probs, raw_returns, states, value_net):
    log_probs_tensor = torch.stack(log_probs).view(-1)
    states_tensor = torch.tensor(np.array(states), dtype=torch.float32)
    values = value_net(states_tensor)
    advantages = raw_returns - values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    assert log_probs_tensor.shape == advantages.shape
    policy_loss = -(log_probs_tensor * advantages).mean()
    value_loss = F.mse_loss(values, raw_returns)
    return policy_loss, value_loss

@dataclass
class ReinforceConfig:
    gamma: float
    hidden_dim: int
    lr_policy: float
    lr_value: float

class REINFORCEAgent:
    def __init__(self, obs_dim, n_actions, config: ReinforceConfig):
        self.config = config
        self.policy_net = PolicyNetwork(obs_dim, n_actions, config.hidden_dim)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config.lr_policy)

    def select_action(self, state):
        return select_action(self.policy_net, state)

    def update_from_episode(self, log_probs, rewards, states=None):
        returns = compute_returns(rewards, self.config.gamma)
        loss = reinforce_loss(log_probs, returns)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return {"policy_loss": loss.item(), "value_loss": None}

class REINFORCEBaselineAgent(REINFORCEAgent):
    def __init__(self, obs_dim, n_actions, config: ReinforceConfig):
        super().__init__(obs_dim, n_actions, config)
        self.value_net = ValueNetwork(obs_dim, config.hidden_dim)
        self.value_optimizer = optim.Adam(self.value_net.parameters(), lr=config.lr_value)

    def update_from_episode(self, log_probs, rewards, states=None):
        raw_returns = compute_raw_returns(rewards, self.config.gamma)
        policy_loss, value_loss = reinforce_baseline_loss(log_probs, raw_returns, states, self.value_net)
        self.optimizer.zero_grad()
        self.value_optimizer.zero_grad()
        (policy_loss + value_loss).backward()
        self.optimizer.step()
        self.value_optimizer.step()
        return {"policy_loss": policy_loss.item(), "value_loss": value_loss.item()}


torch.manual_seed(42)
policy_net_test = PolicyNetwork(obs_dim=4, n_actions=2)
value_net_test = ValueNetwork(obs_dim=4)
dummy_log_probs = [torch.tensor(-0.7 - 0.03 * t, requires_grad=True) for t in range(10)]
dummy_returns = compute_returns([1.0] * 10)
dummy_raw_returns = compute_raw_returns([1.0] * 10)
dummy_states = [np.random.default_rng(42 + t).normal(size=4) for t in range(10)]

loss_r = reinforce_loss(dummy_log_probs, dummy_returns)
dummy_log_probs2 = [torch.tensor(-0.7 - 0.03 * t, requires_grad=True) for t in range(10)]
loss_p, loss_v = reinforce_baseline_loss(dummy_log_probs2, dummy_raw_returns, dummy_states, value_net_test)

print(f"Loss REINFORCE             : {loss_r.item():.6f}")
print(f"Loss politique (+ baseline): {loss_p.item():.6f}")
print(f"Loss valeur    (+ baseline): {loss_v.item():.6f}")
print(f"\nValueNetwork : {sum(p.numel() for p in value_net_test.parameters()):,} parametres")

In [ ]:
# --- Demo de reduction de variance : normes des gradients ---
# On compare la norme du gradient de la loss REINFORCE vs REINFORCE+baseline
# sur 80 episodes simules, pour la meme politique.

torch.manual_seed(42)
np.random.seed(42)

policy_reinforce = PolicyNetwork(obs_dim=4, n_actions=2)
policy_baseline = PolicyNetwork(obs_dim=4, n_actions=2)
policy_baseline.load_state_dict(policy_reinforce.state_dict())
value_net_demo = ValueNetwork(obs_dim=4)

opt_v_demo = optim.Adam(value_net_demo.parameters(), lr=1e-3)
env_var = gym.make("CartPole-v1")
n_demo_episodes = 80
grad_norms_reinforce = []
grad_norms_baseline = []

for ep in range(n_demo_episodes):
    obs, _ = env_var.reset(seed=ep)
    log_probs_r, log_probs_b, rewards_ep, states_ep = [], [], [], []
    done = False

    while not done:
        states_ep.append(obs.copy())
        a_r, lp_r = select_action(policy_reinforce, obs)
        a_b, lp_b = select_action(policy_baseline, obs)
        # On utilise la meme action pour les deux (pour comparer a iso-trajectoire)
        obs, reward, terminated, truncated, _ = env_var.step(a_r)
        # Recalculer log_prob pour policy_baseline avec la meme action
        state_t = torch.tensor(states_ep[-1], dtype=torch.float32).unsqueeze(0)
        lp_b = Categorical(logits=policy_baseline(state_t)).log_prob(torch.tensor(a_r))
        log_probs_r.append(lp_r)
        log_probs_b.append(lp_b)
        rewards_ep.append(reward)
        done = terminated or truncated

    returns_ep = compute_returns(rewards_ep)
    raw_returns_ep = compute_raw_returns(rewards_ep)

    # Gradient REINFORCE
    loss_r = reinforce_loss(log_probs_r, returns_ep)
    policy_reinforce.zero_grad()
    loss_r.backward()
    norm_r = sum(p.grad.norm().item() ** 2 for p in policy_reinforce.parameters() if p.grad is not None) ** 0.5
    grad_norms_reinforce.append(norm_r)

    # Gradient REINFORCE + baseline
    loss_p, loss_v = reinforce_baseline_loss(log_probs_b, raw_returns_ep, states_ep, value_net_demo)
    policy_baseline.zero_grad()
    loss_p.backward()
    norm_b = sum(p.grad.norm().item() ** 2 for p in policy_baseline.parameters() if p.grad is not None) ** 0.5
    grad_norms_baseline.append(norm_b)
    # Entrainer V(s) pour que la baseline devienne utile au fil des episodes
    opt_v_demo.zero_grad()
    loss_v.backward()
    opt_v_demo.step()

env_var.close()

print(f"Norme de gradient moyenne (REINFORCE)          : {np.mean(grad_norms_reinforce):.4f}")
print(f"Norme de gradient moyenne (REINFORCE+baseline) : {np.mean(grad_norms_baseline):.4f}")
print(f"Ecart-type (REINFORCE)          : {np.std(grad_norms_reinforce):.4f}")
print(f"Ecart-type (REINFORCE+baseline) : {np.std(grad_norms_baseline):.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(grad_norms_reinforce, alpha=0.7, color="steelblue", label=f"REINFORCE (std={np.std(grad_norms_reinforce):.3f})")
ax.plot(grad_norms_baseline, alpha=0.7, color="coral", label=f"REINFORCE+baseline (std={np.std(grad_norms_baseline):.3f})")
ax.set_xlabel("Episode")
ax.set_ylabel("Norme du gradient")
ax.set_title("Variance des gradients : REINFORCE vs REINFORCE + baseline")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Convergence des losses sur un batch fixe ---
torch.manual_seed(42)
np.random.seed(42)

pol_r = PolicyNetwork(obs_dim=4, n_actions=2)
pol_b = PolicyNetwork(obs_dim=4, n_actions=2)
pol_b.load_state_dict(pol_r.state_dict())
val_b = ValueNetwork(obs_dim=4)

opt_r = optim.Adam(pol_r.parameters(), lr=1e-3)
opt_pb = optim.Adam(pol_b.parameters(), lr=1e-3)
opt_vb = optim.Adam(val_b.parameters(), lr=1e-3)

# Simuler 20 episodes courts pour comparer les losses
env_conv = gym.make("CartPole-v1")
losses_r_conv = []
losses_p_conv = []
losses_v_conv = []

for ep in range(20):
    obs, _ = env_conv.reset(seed=ep * 7)
    lp_r, lp_b, rews, sts = [], [], [], []
    done = False
    while not done:
        sts.append(obs.copy())
        a, lp = select_action(pol_r, obs)
        state_t = torch.tensor(sts[-1], dtype=torch.float32).unsqueeze(0)
        lp2 = Categorical(logits=pol_b(state_t)).log_prob(torch.tensor(a))
        obs, rew, term, trunc, _ = env_conv.step(a)
        lp_r.append(lp)
        lp_b.append(lp2)
        rews.append(rew)
        done = term or trunc

    rets = compute_returns(rews)
    raw_rets = compute_raw_returns(rews)

    # REINFORCE
    loss_r = reinforce_loss(lp_r, rets)
    opt_r.zero_grad()
    loss_r.backward()
    opt_r.step()
    losses_r_conv.append(abs(loss_r.item()))

    # REINFORCE + baseline
    loss_p, loss_v = reinforce_baseline_loss(lp_b, raw_rets, sts, val_b)
    opt_pb.zero_grad()
    opt_vb.zero_grad()
    (loss_p + loss_v).backward()
    opt_pb.step()
    opt_vb.step()
    losses_p_conv.append(abs(loss_p.item()))
    losses_v_conv.append(loss_v.item())

env_conv.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(losses_r_conv, color="steelblue", linewidth=2, label="REINFORCE")
ax1.plot(losses_p_conv, color="coral", linewidth=2, linestyle="--", label="REINFORCE+baseline (policy)")
ax1.set_xlabel("Episode")
ax1.set_ylabel("|Loss|")
ax1.set_title("Loss de politique")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(losses_v_conv, color="green", linewidth=2, label="Loss valeur (MSE)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("MSE")
ax2.set_title("Loss du reseau de valeur (REINFORCE+baseline)")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### La baseline devient efficace progressivement

La démonstration précédente entraîne $V(s)$ au fil des épisodes. Au début, la baseline est presque arbitraire, donc la réduction de variance reste limitée. Plus le réseau de valeur apprend à approximer les returns bruts, plus l'avantage centré $G_t - V(s_t)$ devient informatif.

Cette amélioration ne change pas l'objectif optimisé par la politique : elle rend simplement le signal de gradient moins bruyant. C'est aussi pour cela qu'on ajoute maintenant un **diagnostic d'entropie** dans l'entraînement complet : si la politique se fige trop tôt, la baseline ne suffira pas à sauver l'exploration.

## Les deux algorithmes complets

À ce stade, la boucle d'environnement doit rester très lisible : elle collecte une trajectoire
complète, puis confie la mise à jour mathématique à l'agent. C'est exactement la séparation que l'on
veut pour REINFORCE : l'environnement produit des épisodes, l'agent transforme ces épisodes en
gradient de politique.

### REINFORCE sans baseline

$$
\boxed{
\begin{aligned}
&\textbf{Initialiser } \pi_\theta(a\mid s),\ \text{optimiseur Adam}\\[1mm]
&\textbf{pour } episode = 1,\dots,N \textbf{ faire}\\
&\quad s_0 \leftarrow env.reset()\\
&\quad \mathcal{D} \leftarrow \varnothing\\[1mm]
&\quad \textbf{tant que l'épisode n'est pas terminé faire}\\
&\quad\quad a_t \sim \pi_\theta(\cdot\mid s_t)\\
&\quad\quad \ell_t \leftarrow \log \pi_\theta(a_t\mid s_t)\\
&\quad\quad s_{t+1}, r_{t+1}, done \leftarrow env.step(a_t)\\
&\quad\quad \mathcal{D} \leftarrow \mathcal{D}\cup\{s_t,a_t,\ell_t,r_{t+1}\}\\
&\quad\quad s_t \leftarrow s_{t+1}\\
&\quad \textbf{fin tant que}\\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{Retour Monte-Carlo depuis chaque temps}}\\
&\quad G_t \leftarrow \sum_{k=t}^{T-1}\gamma^{k-t}r_{k+1}\\
&\quad \hat G_t \leftarrow \frac{G_t-\mu(G)}{\sigma(G)+\varepsilon}\\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{Gradient de politique : augmenter les actions à bon retour}}\\
&\quad \mathcal{L}_{\pi}(\theta)
    \leftarrow -\frac{1}{T}\sum_{t=0}^{T-1}\ell_t\,\hat G_t\\
&\quad \theta \leftarrow \theta - \alpha\nabla_\theta \mathcal{L}_{\pi}(\theta)\\
&\quad \text{enregistrer reward, loss, entropie}\\
&\textbf{fin pour}
\end{aligned}
}
$$

### REINFORCE avec baseline

$$
\boxed{
\begin{aligned}
&\textbf{Initialiser } \pi_\theta(a\mid s),\ V_\psi(s),\ \text{optimiseurs Adam}\\[1mm]
&\textbf{pour } episode = 1,\dots,N \textbf{ faire}\\
&\quad \text{collecter une trajectoire complète }
        \mathcal{D}=\{s_t,a_t,\log\pi_\theta(a_t\mid s_t),r_{t+1}\}_{t=0}^{T-1}\\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{Cible Monte-Carlo pour le critique}}\\
&\quad G_t \leftarrow \sum_{k=t}^{T-1}\gamma^{k-t}r_{k+1}\\
&\quad V_t \leftarrow V_\psi(s_t)\\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{Avantage : retour observé moins valeur attendue}}\\
&\quad A_t \leftarrow G_t - V_t\\
&\quad \hat A_t \leftarrow \frac{A_t-\mu(A)}{\sigma(A)+\varepsilon}\\[1mm]
&\quad \textcolor{gray}{\triangleright\ \text{L'acteur utilise l'avantage, le critique régresse vers }G_t}\\
&\quad \mathcal{L}_{actor}(\theta)
    \leftarrow -\frac{1}{T}\sum_{t=0}^{T-1}
    \log\pi_\theta(a_t\mid s_t)\,\hat A_t\\
&\quad \mathcal{L}_{critic}(\psi)
    \leftarrow \frac{1}{T}\sum_{t=0}^{T-1}
    \bigl(V_\psi(s_t)-G_t\bigr)^2\\
&\quad \theta \leftarrow \theta-\alpha_\pi\nabla_\theta\mathcal{L}_{actor}\\
&\quad \psi \leftarrow \psi-\alpha_V\nabla_\psi\mathcal{L}_{critic}\\
&\quad \text{enregistrer reward, losses, entropie}\\
&\textbf{fin pour}
\end{aligned}
}
$$

**Lecture.** La version sans baseline utilise directement le retour Monte-Carlo comme poids du
gradient. La version avec baseline remplace ce poids par un avantage $A_t=G_t-V(s_t)$ : on ne demande
plus seulement si le retour était bon en absolu, mais s'il était meilleur que ce que l'on pouvait
attendre depuis cet état. C'est cette idée qui prépare naturellement le passage à Actor-Critic.

## Hyperparametres

On fixe **exactement les memes valeurs** pour les deux algorithmes afin d'isoler l'effet de la baseline. La seule variable est la presence ou non du reseau de valeur.

| Parametre | Valeur | Role |
|-----------|--------|------|
| `hidden_dim` | 64 | Taille des couches cachees |
| `lr_policy` | 1e-3 | Taux d'apprentissage du reseau politique (Adam) |
| `lr_value` | 1e-3 | Taux d'apprentissage du reseau de valeur (Adam) |
| `gamma` | 0.99 | Facteur d'actualisation |
| `n_episodes` | 1000 | Nombre d'episodes d'entrainement |
| `seed` | 42 | Graine aleatoire (meme pour les deux) |

Pas de replay buffer, pas d'epsilon, pas de target network : REINFORCE est Monte Carlo on-policy, chaque episode est traite individuellement.

On entraîne en mode headless : le rendu visuel est séparé de l'entraînement pour éviter de ralentir la boucle et pour garder le notebook exécutable sans fenêtre graphique. La démonstration finale enregistrera une courte vidéo `rgb_array` de la politique, affichée directement dans le notebook.

In [ ]:
# Configuration commune : seule la présence de la baseline change entre les deux runs.
SEED = 42
ENV_ID = "CartPole-v1"
RUN_REINFORCE = True
RUN_REINFORCE_BASELINE = True
EVAL_SEED = 0

HIDDEN_DIM = 64
LR_POLICY = 1e-3
LR_VALUE = 1e-3
GAMMA = 0.99
N_EPISODES = 500

torch.manual_seed(SEED)
np.random.seed(SEED)

config = ReinforceConfig(
    gamma=GAMMA,
    hidden_dim=HIDDEN_DIM,
    lr_policy=LR_POLICY,
    lr_value=LR_VALUE,
)

In [ ]:
def episode_entropy(policy_net, states):
    if not states:
        return 0.0
    with torch.no_grad():
        states_tensor = torch.tensor(np.array(states), dtype=torch.float32)
        logits = policy_net(states_tensor)
        dist = Categorical(logits=logits)
        return float(dist.entropy().mean().item())


def train_reinforce_agent(agent, label, use_states=False):
    env = gym.make(ENV_ID)
    rewards_history = []
    policy_losses = []
    value_losses = []
    entropies = []

    for episode in range(N_EPISODES):
        obs, _ = env.reset()
        log_probs, rewards, states = [], [], []
        done = False

        while not done:
            if use_states:
                states.append(obs.copy())
            else:
                states.append(obs.copy())
            action, log_prob = agent.select_action(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            done = terminated or truncated

        metrics = agent.update_from_episode(log_probs, rewards, states if use_states else None)
        rewards_history.append(sum(rewards))
        policy_losses.append(metrics["policy_loss"])
        value_losses.append(metrics["value_loss"])
        entropies.append(episode_entropy(agent.policy_net, states))

        if episode % 50 == 0 or episode == N_EPISODES - 1:
            avg = np.mean(rewards_history[-50:]) if len(rewards_history) >= 50 else np.mean(rewards_history)
            value_msg = "" if metrics["value_loss"] is None else f" | Loss V : {metrics['value_loss']:.4f}"
            print(f"Episode {episode:5d} | Recompense : {sum(rewards):6.1f} | Moy(50) : {avg:6.1f} | Loss pi : {metrics['policy_loss']:.4f}{value_msg} | H : {entropies[-1]:.4f}")

    env.close()
    return rewards_history, policy_losses, value_losses, entropies

In [ ]:
print("=" * 60)
print("ENTRAINEMENT REINFORCE")
print("=" * 60)

torch.manual_seed(SEED)
np.random.seed(SEED)

env_probe = gym.make(ENV_ID)
obs_dim = env_probe.observation_space.shape[0]
n_actions = env_probe.action_space.n
env_probe.close()

reinforce_agent = REINFORCEAgent(obs_dim, n_actions, config)
if RUN_REINFORCE:
    reinforce_rewards, reinforce_losses, reinforce_value_losses, reinforce_entropies = train_reinforce_agent(
        reinforce_agent,
        "REINFORCE",
        use_states=False
    )
else:
    reinforce_rewards, reinforce_losses, reinforce_value_losses, reinforce_entropies = [], [], [], []
policy_r_trained = reinforce_agent.policy_net
print("\nREINFORCE termine.")

In [ ]:
print("=" * 60)
print("ENTRAINEMENT REINFORCE + BASELINE")
print("=" * 60)

torch.manual_seed(SEED)
np.random.seed(SEED)

baseline_agent = REINFORCEBaselineAgent(obs_dim, n_actions, config)
if RUN_REINFORCE_BASELINE:
    baseline_rewards, baseline_policy_losses, baseline_value_losses, baseline_entropies = train_reinforce_agent(
        baseline_agent,
        "REINFORCE + BASELINE",
        use_states=True
    )
else:
    baseline_rewards, baseline_policy_losses, baseline_value_losses, baseline_entropies = [], [], [], []
policy_b_trained = baseline_agent.policy_net
print("\nREINFORCE + Baseline termine.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
window = 50

ax = axes[0]
ax.plot(reinforce_rewards, alpha=0.2, color="steelblue")
ax.plot(baseline_rewards, alpha=0.2, color="coral")
if len(reinforce_rewards) >= window:
    ma_r = np.convolve(reinforce_rewards, np.ones(window) / window, mode="valid")
    ma_b = np.convolve(baseline_rewards, np.ones(window) / window, mode="valid")
    x = range(window - 1, window - 1 + len(ma_r))
    ax.plot(x, ma_r, color="steelblue", linewidth=2, label=f"REINFORCE (moy. mobile {window})")
    ax.plot(x, ma_b, color="coral", linewidth=2, label=f"REINFORCE+baseline (moy. mobile {window})")
ax.axhline(500, color="gray", linestyle=":", linewidth=1, alpha=0.7, label="Score max")
ax.set_xlabel("Episode")
ax.set_ylabel("Recompense")
ax.set_title("Courbe d'apprentissage")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(reinforce_losses, alpha=0.2, color="steelblue")
ax.plot(baseline_policy_losses, alpha=0.2, color="coral")
w2 = 20
if len(reinforce_losses) >= w2:
    ma_lr = np.convolve(reinforce_losses, np.ones(w2) / w2, mode="valid")
    ma_lb = np.convolve(baseline_policy_losses, np.ones(w2) / w2, mode="valid")
    x2 = range(w2 - 1, w2 - 1 + len(ma_lr))
    ax.plot(x2, ma_lr, color="steelblue", linewidth=2, label="REINFORCE")
    ax.plot(x2, ma_lb, color="coral", linewidth=2, label="REINFORCE+baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Loss politique")
ax.set_title("Loss du reseau de politique")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
value_series = np.array([np.nan if v is None else v for v in baseline_value_losses], dtype=float)
ax.plot(value_series, alpha=0.3, color="green")
valid_value_series = value_series[~np.isnan(value_series)]
if len(valid_value_series) >= w2:
    ma_lv = np.convolve(valid_value_series, np.ones(w2) / w2, mode="valid")
    x3 = range(w2 - 1, w2 - 1 + len(ma_lv))
    ax.plot(x3, ma_lv, color="green", linewidth=2, label="Loss valeur (MSE)")
ax.set_xlabel("Episode")
ax.set_ylabel("MSE")
ax.set_title("Loss du reseau de valeur V(s)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[3]
ax.plot(reinforce_entropies, color="steelblue", linewidth=2, label="REINFORCE")
ax.plot(baseline_entropies, color="coral", linewidth=2, linestyle="--", label="REINFORCE+baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Entropie moyenne")
ax.set_title("Diagnostic d'exploration")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analyse des courbes

### Courbe de recompense
Les deux méthodes finissent par résoudre CartPole, mais la baseline réduit souvent le temps passé dans les épisodes très irréguliers. Ce n'est pas une magie d'optimisation : c'est un meilleur estimateur du signal de gradient.

### Loss de politique
La loss de politique reste bruyante parce qu'elle dépend d'épisodes complets. Il faut donc la lire comme un reflet de la direction du gradient, pas comme une courbe de convergence lisse à la manière d'une régression supervisée.

### Loss de valeur
La MSE du critique raconte si `V(s)` apprend à suivre l'échelle des returns bruts. C'est pour cela qu'on n'utilise pas les returns normalisés comme cible du réseau de valeur.

### Entropie
L'entropie moyenne permet de voir si la politique se fige trop tôt. Une entropie qui s'écroule très vite signale une politique devenue quasi déterministe avant d'avoir assez exploré. Ici, elle joue le rôle d'un diagnostic, pas d'un terme ajouté à l'objectif.

In [ ]:
# --- Metriques finales ---
def print_metrics(name, rewards, threshold=400):
    last_50 = rewards[-50:]
    reached = None
    for i in range(50, len(rewards)):
        if np.mean(rewards[i - 50:i]) >= threshold:
            reached = i
            break
    print(f"  Moy. 50 derniers episodes : {np.mean(last_50):.1f}")
    print(f"  Meilleure recompense      : {max(rewards):.1f}")
    print(f"  Recompense finale         : {rewards[-1]:.1f}")
    print(f"  Episodes pour atteindre {threshold} (moy. mobile) : {reached if reached else 'non atteint'}")


print("REINFORCE :")
print_metrics("REINFORCE", reinforce_rewards)
print(f"\nREINFORCE + Baseline :")
print_metrics("REINFORCE+Baseline", baseline_rewards)

## Tableau recapitulatif

| Dimension | REINFORCE | REINFORCE + Baseline |
|-----------|-----------|----------------------|
| Architecture politique | `PolicyNetwork(4, 2)` | `PolicyNetwork(4, 2)` (identique) |
| Reseau de valeur | non | `ValueNetwork(4)` |
| Returns utilises | $\hat{G}_t$ (normalises) | $\hat{G}_t$ (identiques) |
| Ponderation du gradient | $\hat{G}_t$ | $A_t = \hat{G}_t - V(s_t)$ |
| Loss de valeur | non | $\text{MSE}(G_t, V(s_t))$ |
| Nombre d'optimiseurs | 1 | 2 |
| Variance du gradient | elevee | reduite |
| Vitesse de convergence | plus lente | generalement plus rapide |
| Stabilite | variable | plus reguliere |
| Complexite code | simple | moderee (+10 lignes) |

La reduction de variance est le seul apport de la baseline. Sur CartPole (espace simple, 2 actions), l'avantage est visible mais modeste. Sur des environnements plus complexes (MuJoCo, Atari), la difference de stabilite est nette.

**Note.** REINFORCE + baseline est donc le "bébé" de la famille **Actor-Critic**, l'ancetre direct de l'algorithme **A2C** (Advantage Actor-Critic), qui remplace le return Monte Carlo par un bootstrap TD et partage une partie de l'architecture entre reseau politique et reseau de valeur.

## Policy gradient vs value-based : bilan

| Dimension | Q-learning / DQN | REINFORCE / REINFORCE+Baseline |
|-----------|------------------|--------------------------------|
| Signal d'apprentissage | Cible TD locale | Return d'épisode complet |
| Politique pendant l'entraînement | ε-greedy | Stochastique par construction |
| Source principale de variance | Bootstrap + approximation | Monte Carlo |
| Remède principal | Replay + target network | Baseline apprise |
| Évaluation standard | Greedy | Stochastique d'abord, greedy en complément |

Le point important pour la suite est le suivant : une politique gradient se comprend d'abord comme une **distribution**. La figer en argmax reste utile pour l'inspection, mais ce n'est pas la référence principale de performance pendant l'apprentissage.

In [ ]:
def evaluate_and_display_agent(
    policy_net,
    label,
    n_episodes=5,
    mode="stochastic",
    seed=EVAL_SEED,
    video_root="videos/03_reinforce_cartpole",
):
    """Évalue une politique et affiche le dernier replay vidéo enregistré."""
    rewards = []
    safe_label = label.lower().replace(" ", "_").replace("-", "_").replace("+", "plus")
    video_dir = Path(video_root) / safe_label / mode
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_{mode}",
    )

    policy_net.eval()

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0
            done = False

            while not done:
                with torch.no_grad():
                    obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                    logits = policy_net(obs_t)
                    if mode == "greedy":
                        action = int(torch.argmax(logits, dim=-1).item())
                    else:
                        action = int(Categorical(logits=logits).sample().item())

                obs, reward, terminated, truncated, _ = env.step(action)
                total += reward
                done = terminated or truncated

            rewards.append(total)
            print(f"[{label} | {mode}] Épisode {ep + 1} : {total:.0f} pas")

    finally:
        env.close()

    print(f"[{label} | {mode}] Moyenne : {np.mean(rewards):.1f} pas sur {n_episodes} épisodes")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if videos and Video is not None and display is not None:
        print(f"Replay vidéo {label} ({mode}) :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    elif not videos:
        print(f"Aucune vidéo générée pour {label} ({mode}).")
    else:
        print(f"Vidéo générée mais affichage IPython indisponible : {videos[-1]}")

    return rewards


print("=== REINFORCE stochastic ===")
reinforce_demo = evaluate_and_display_agent(policy_r_trained, "REINFORCE", mode="stochastic")

print("=== REINFORCE + Baseline stochastic ===")
baseline_demo = evaluate_and_display_agent(policy_b_trained, "REINFORCE+Baseline", mode="stochastic")

print("=== REINFORCE greedy (complément) ===")
reinforce_demo_greedy = evaluate_and_display_agent(policy_r_trained, "REINFORCE", mode="greedy")

print("=== REINFORCE + Baseline greedy (complément) ===")
baseline_demo_greedy = evaluate_and_display_agent(policy_b_trained, "REINFORCE+Baseline", mode="greedy")


## Conclusion et ponts vers la suite

REINFORCE marque le passage des méthodes value-based aux **policy gradients**. L'agent ne choisit plus
ses actions en maximisant une table ou un réseau $Q(s,a)$ : il apprend directement les paramètres
d'une politique stochastique $\pi_\theta(a\mid s)$. Les actions qui ont mené à de bons retours voient
leur probabilité augmenter ; celles qui ont mené à de mauvais retours deviennent moins probables.

Cette idée est très générale, mais elle a un prix : le signal d'apprentissage est bruité. Un épisode
réussi ou raté peut dépendre de beaucoup de décisions mélangées, et le retour Monte-Carlo arrive
seulement à la fin. La baseline améliore déjà nettement la situation : en soustrayant une estimation
de la valeur de l'état, on apprend à partir d'un **avantage** plutôt qu'à partir du retour brut.

La suite naturelle est donc **Actor-Critic**. L'acteur gardera le gradient de politique, tandis qu'un
critique apprendra à prédire la valeur pour fournir un signal plus fréquent, moins variable et
bootstrappé. C'est le point de rencontre entre les idées de REINFORCE et celles du TD learning.